In [1]:
import itertools
import numpy as np

from Util.Problems import Problem, solution
import Util.math_functions as mathf

class P024(Problem):
    number = 24
    title = "Lexicographic Permutations"
    description = """<p>A permutation is an ordered arrangement of objects. For example, 3124 is one possible permutation of the digits 1, 2, 3 and 4. If all of the permutations are listed numerically or alphabetically, we call it lexicographic order. The lexicographic permutations of 0, 1 and 2 are:</p><p class="center">012   021   102   120   201   210</p><p>What is the millionth lexicographic permutation of the digits 0, 1, 2, 3, 4, 5, 6, 7, 8 and 9?</p>"""
    min_digit = 0
    max_digit = 9
    permutation_index = 1000000

In [2]:
p = P024()
p.describe()

## Problem 24: Lexicographic Permutations

<p>A permutation is an ordered arrangement of objects. For example, 3124 is one possible permutation of the digits 1, 2, 3 and 4. If all of the permutations are listed numerically or alphabetically, we call it lexicographic order. The lexicographic permutations of 0, 1 and 2 are:</p><p class="center">012   021   102   120   201   210</p><p>What is the millionth lexicographic permutation of the digits 0, 1, 2, 3, 4, 5, 6, 7, 8 and 9?</p>

### Solution notes
My initial idea is to calculate only the desired permutation. This could be done by taking its index, and looking at the options starting with the first digit. In this particular example, we have 10 total digits. The first digit could be any of these 10. Given the first digit, the remaining 9 digits can be ordered in $9!$ ways. By calculating $\frac{permutation\_index}{9!}$, we get the first digit of the permutation. In this example, we are looking for the permutation with permutation index $1.000.000$. $9! = 362.880$, so at index $1.000.000$, we have passed all permutations starting with $0$ and $1$, as these make up the first $2 \times 362.880$ permutations. The desired index is less than $3 \times 362.880$, though, so we know the first digit is a 2. We can repeat this process for the second digit, looking at how many permutations there are for the remaining 8 digits after the first 2 have been placed etc. Due to an off-by-one error in my indexing, this method yielded the wrong results, so I implemented a very slow brute force, to find out the correct answer and used that to fix this implementation.

In [3]:
@solution(P024, first=True, make_fast=False, warmup_args=(P024.min_digit, P024.max_digit, P024.permutation_index))
def index_calculation(min_digit, max_digit, permutation_index):
    number_of_items = max_digit - min_digit + 1
    running_index = permutation_index - 1
    available_digits = np.ones(number_of_items, dtype=np.bool_)
    permutation = np.empty(number_of_items, dtype=np.int8)
    for i in range(1, number_of_items):
        after_next_digit = mathf.factorial(number_of_items - i)
        digit_index = running_index // after_next_digit
        running_index = running_index % after_next_digit
        digit = available_digits.nonzero()[0][digit_index]
        available_digits[digit] = 0
        permutation[i - 1] = digit
    permutation[-1] = available_digits.nonzero()[0][0]
    result = ""
    for permutation_digit in permutation:
        result += str(permutation_digit)
    return result

In [4]:
p.test_all()

2783915460 found after 1000 tests in 0.006900 ms by index_calculation (first)


As mentioned, I needed to know the answer to bugfix my fast solution, so I wrote this horribly slow non-numba brute force implementation which creates a list of all permutations using itertools and returns the one at index $999.999$

In [5]:
@solution(P024, max_tests=1, make_fast=False, warmup_args=(P024.min_digit, P024.max_digit, P024.permutation_index))
def brute_force(min_digit, max_digit, permutation_index):
    digits = [digit for digit in range(min_digit, max_digit + 1)]
    permutations = list(itertools.permutations(digits))
    return ''.join(str(digit) for digit in permutations[permutation_index - 1])

In [6]:
p.test_all()

2783915460 found after 1 test in 382.628600 ms by brute_force
2783915460 found after 1000 tests in 0.007163 ms by index_calculation (first)
